# Check skill name completeness
Check completeness of the skills name in row 1700 and before because that was when the code was not corrected

to be safe, we will check all rows

In [13]:
import pandas as pd

course_df = pd.read_csv(
    "data/2025-2026_module_clean_with_prereq_skillsfuture.csv",
    usecols=["extracted_skills"],
    encoding="utf-8-sig",
)

taxo_df = pd.read_csv(
    "data/skills_taxo.csv",
    usecols=["parent_skill_title"],
    encoding="utf-8-sig",
)

taxonomy_titles = {
    str(value).strip()
    for value in taxo_df["parent_skill_title"].dropna()
    if str(value).strip()
}

unmatched_skills_df = pd.DataFrame(
    {
        "unmatched_skill": sorted(
            {
                skill
                for raw_value in course_df["extracted_skills"].dropna()
                for skill in [token.strip() for token in str(raw_value).split("|")]
                if skill and skill not in taxonomy_titles
            }
        )
    }
)

print(f"Unique unmatched skills: {len(unmatched_skills_df)}")
unmatched_skills_df


Unique unmatched skills: 26


,unmatched_skill
0,Coaching and Assessment Management
1,Connection and Measurement
2,Cultural
3,Drainage and Gas Systems Design
4,Electrical
5,Electrical Termination
6,Electronic and Control Engineering
7,Environmental Management System Policies
8,Ethics
9,Generative AI Innovation


In [14]:
taxo_titles = (
    pd.read_csv(
        "data/skills_taxo.csv",
        usecols=["parent_skill_title"],
        encoding="utf-8-sig",
    )["parent_skill_title"]
    .dropna()
    .astype(str)
    .str.strip()
)

taxo_titles = [title for title in taxo_titles if title]

def find_prefix_comma_matches(skill):
    prefix = f"{skill},"
    matches = [title for title in taxo_titles if title.startswith(prefix)]
    return " | ".join(matches) if matches else None

unmatched_skills_df["matched_parent_skill_title"] = unmatched_skills_df["unmatched_skill"].apply(
    find_prefix_comma_matches
)

unmatched_skills_df


,unmatched_skill,matched_parent_skill_title
0,Coaching and Assessment Management,None
1,Connection and Measurement,None
2,Cultural,"Cultural, Heritage and Socio-economic Sensitiv..."
3,Drainage and Gas Systems Design,None
4,Electrical,"Electrical, Electronic and Control Engineering"
5,Electrical Termination,"Electrical Termination, Connection and Measure..."
6,Electronic and Control Engineering,None
7,Environmental Management System Policies,"Environmental Management System Policies, Stan..."
8,Ethics,"Ethics, Values and Legislation"
9,Generative AI Innovation,"Generative AI Innovation, Research and Develop..."


In [ ]:
# Check if any matched parent skill titles contain " | "
(unmatched_skills_df["matched_parent_skill_title"].fillna("").str.contains(r" \| ")).any()


np.False_

for any unmatched skill in the data frame, 
    
    if they are found in the 2025-2026_module_clean_with_prereq_skillsfuture, then replace it 
    
    if there is a matched_parent_skill_title, if not then remove it

In [17]:
import pandas as pd

course_df = pd.read_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv")
# unmatched_skills_df already exists

replacement_map = (
    unmatched_skills_df.loc[
        unmatched_skills_df["matched_parent_skill_title"].notna()
        & unmatched_skills_df["matched_parent_skill_title"].astype(str).str.strip().ne(""),
        ["unmatched_skill", "matched_parent_skill_title"],
    ]
    .drop_duplicates(subset=["unmatched_skill"])
    .set_index("unmatched_skill")["matched_parent_skill_title"]
    .to_dict()
)

remove_set = set(
    unmatched_skills_df.loc[
        unmatched_skills_df["matched_parent_skill_title"].isna()
        | unmatched_skills_df["matched_parent_skill_title"].astype(str).str.strip().eq(""),
        "unmatched_skill",
    ]
)

def clean_extracted_skills(raw_text):
    if pd.isna(raw_text) or not str(raw_text).strip():
        return raw_text

    tokens = [token.strip() for token in str(raw_text).split("|") if token.strip()]
    cleaned = []

    for token in tokens:
        if token in replacement_map:
            cleaned.append(replacement_map[token])
        elif token in remove_set:
            continue
        else:
            cleaned.append(token)

    # remove duplicates while preserving order
    cleaned = list(dict.fromkeys(cleaned))
    return " | ".join(cleaned)

course_df["extracted_skills"] = course_df["extracted_skills"].apply(clean_extracted_skills)



In [18]:
course_df.head()

,moduleCode,title,description,prereqCourseCodes,extracted_skills,extracted_apps_tools,done
0,ACC1701,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
1,ACC1701A,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
2,ACC1701B,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
3,ACC1701C,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
4,ACC1701D,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...


In [19]:
course_df.to_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv", index=False)


Check that there is no more unmatched skills

In [20]:
course_df = pd.read_csv(
    "data/2025-2026_module_clean_with_prereq_skillsfuture.csv",
    usecols=["extracted_skills"],
    encoding="utf-8-sig",
)

taxo_df = pd.read_csv(
    "data/skills_taxo.csv",
    usecols=["parent_skill_title"],
    encoding="utf-8-sig",
)

taxonomy_titles = {
    str(value).strip()
    for value in taxo_df["parent_skill_title"].dropna()
    if str(value).strip()
}

unmatched_skills_df = pd.DataFrame(
    {
        "unmatched_skill": sorted(
            {
                skill
                for raw_value in course_df["extracted_skills"].dropna()
                for skill in [token.strip() for token in str(raw_value).split("|")]
                if skill and skill not in taxonomy_titles
            }
        )
    }
)

print(f"Unique unmatched skills: {len(unmatched_skills_df)}")
unmatched_skills_df


Unique unmatched skills: 0


,unmatched_skill


# Check rows without skills & apps tools

In [17]:
import pandas as pd

df = pd.read_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv")

skills_col = "extracted_skills"
apps_col = "extracted_apps_tools"  # actual column name in the CSV

missing_skills = df[skills_col].fillna("").astype(str).str.strip().eq("")
missing_apps = df[apps_col].fillna("").astype(str).str.strip().eq("")

# If you want rows where both columns are missing
missing_both = df[missing_skills & missing_apps]

print("\nRows missing both extracted_skills and extracted_apps_tools:", len(missing_both))
print(missing_both[["moduleCode", "title", skills_col, apps_col, "done"]])


Rows missing both extracted_skills and extracted_apps_tools: 1271
     moduleCode                                              title  \
78      ALS1020                          Learning to Choose Better   
82       AN2203             Peoples and Cultures of Southeast Asia   
83       AN2204                                 Media Anthropology   
93       AN3208                             Critiquing Development   
96       AN3551     FASS Undergraduate Research Opportunity (UROP)   
...         ...                                                ...   
5422   UTW2001K               Public Memory, Identity and Rhetoric   
5423   UTW2001L                 Visualizing Southeast Asian Cities   
5424   UTW2001M                            Sport and Socialization   
5428   UTW2001S                              Masculinities on Film   
5430   UTW2001W  Alter ego / authentic self? Online political i...   

     extracted_skills extracted_apps_tools                            done  
78             

In [19]:
done_counts = (
    df["done"]
    .fillna("<missing>")
    .astype(str)
    .str.strip()
    .replace("", "<blank>")
    .value_counts()
)

print(done_counts)

done
error: Section 'extracted_apps_and_tools' not found in downloaded CSV.    3928
error: Skills download failed.                                            1245
success                                                                    267
pending                                                                     29
error: Section 'extracted_skills' not found in downloaded CSV.               1
Name: count, dtype: int64


error: Section 'extracted_apps_and_tools' not found in downloaded CSV.

for the Apps & Tools export: the downloaded CSV does not contain a line exactly equal to extracted_apps_and_tools


error: Skills download failed.

This is recorded when the script clicks the Skills tab’s download button but no new CSV appears in ~/Downloads within DOWNLOAD_TIMEOUT = 5 seconds

error: Section 'extracted_skills' not found in downloaded CSV.

This is recorded when a CSV was downloaded for the Skills tab, but that file does not contain a line exactly equal to extracted_skills

I realised that for those with  `error: Section 'extracted_skills' not found in downloaded CSV.` there was a extraction bug so I patched the extraction code and reran for the the rows

## Change applicable back to pending for extraction

In [ ]:
error_value = "error: Section 'extracted_skills' not found in downloaded CSV."

missing_both_mask = (
    df["extracted_skills"].fillna("").astype(str).str.strip().eq("")
    & df["extracted_apps_tools"].fillna("").astype(str).str.strip().eq("")
)

error_mask = df["done"].fillna("").astype(str).str.strip().eq(error_value)

count = (missing_both_mask & error_mask).sum()
print(count)

In [ ]:
import pandas as pd
path = "data/2025-2026_module_clean_with_prereq_skillsfuture.csv"
df = pd.read_csv(path)

error_value = "error: Section 'extracted_skills' not found in downloaded CSV."

mask = (
    df["extracted_skills"].fillna("").astype(str).str.strip().eq("")
    & df["extracted_apps_tools"].fillna("").astype(str).str.strip().eq("")
    & df["done"].fillna("").astype(str).str.strip().eq(error_value)
)

print("Rows to reset to pending:", mask.sum())

df.loc[mask, "done"] = "pending"

In [ ]:
df.loc[mask, ["moduleCode", "title", "extracted_skills", "extracted_apps_tools", "done"]]


In [7]:
df.to_csv(path, index=False)

## Check again

In [22]:
import pandas as pd

df = pd.read_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv")

skills_col = "extracted_skills"
apps_col = "extracted_apps_tools"  # actual column name in the CSV

missing_skills = df[skills_col].fillna("").astype(str).str.strip().eq("")
missing_apps = df[apps_col].fillna("").astype(str).str.strip().eq("")

# If you want rows where both columns are missing
missing_both = df[missing_skills & missing_apps]

print("\nRows missing both extracted_skills and extracted_apps_tools:", len(missing_both))
print(missing_both[["moduleCode", "title", skills_col, apps_col, "done"]])


Rows missing both extracted_skills and extracted_apps_tools: 1242
     moduleCode                                              title  \
78      ALS1020                          Learning to Choose Better   
82       AN2203             Peoples and Cultures of Southeast Asia   
83       AN2204                                 Media Anthropology   
93       AN3208                             Critiquing Development   
96       AN3551     FASS Undergraduate Research Opportunity (UROP)   
...         ...                                                ...   
5413   UTW1001T                         How Rich Should Anyone Be?   
5422   UTW2001K               Public Memory, Identity and Rhetoric   
5423   UTW2001L                 Visualizing Southeast Asian Cities   
5424   UTW2001M                            Sport and Socialization   
5430   UTW2001W  Alter ego / authentic self? Online political i...   

     extracted_skills extracted_apps_tools                            done  
78             

Yep the number of rows with missing skills & apps tools have decreased by 30 as expected

Now we will sample 10 rows, and check if the description retruns no skill and no apps tools

## Sample check

In [23]:
sample_10 = missing_both.sample(n=min(10, len(missing_both)))  # add random_state=42 if you want reproducible results

for _, row in sample_10.iterrows():
    print(f"\n{row['moduleCode']} - {row['title']}")
    print(row["description"])
    print("-" * 80)


CH3261 - Prescribed Text: The Four Books
This is an in-depth evaluation of one to two prescribed texts not covered under CH2261. Significant chapters of the texts will be selected for intensive reading and close analysis. The course will be of interest to students who wish to further their study in Chinese thought, history and literature. Target students for this course are second- and third-year undergraduates across the University and those majoring in Chinese Studies.
--------------------------------------------------------------------------------

DSA1992 - Exchange Enrichment Course
Not Applicable
--------------------------------------------------------------------------------

LAV2201 - Vietnamese 2
This course aims to further enhance students' proficiency in the four basic skills of speaking, listening, reading, and writing. Students will be exposed to more language functions and a wider range of topics. Through reading formulaic authentic texts, students will be introduced to 

Of these 10 rows, 1's description return a skill and the other 9 does not have any skills or apps tools

so assume the extraction was 90% correct for the rows without skill or apps tools 

In [25]:
1242 * 0.1

124.2

In [26]:
124/5470

0.0226691042047532

Only 2% of the all courses might have extraction error, which is quite little

due to time constraints, we will not do any further check/ corrections to them

Furthermore, for this sample the only row that returned a skill was: LAV2201 - Vietnamese 2
This course aims to further enhance students' proficiency in the four basic skills of speaking, listening, reading, and writing. Students will be exposed to more language functions and a wider range of topics. Through reading formulaic authentic texts, students will be introduced to the language in written form as it appears in daily communicative situations to achieve further understanding of the country, its culture and its people. At the end of this course, students will be equipped with a sound foundation of the language to maintain communication on topics relating to their personal and immediate environment.

Basically this is a mod teaching vietnamese language. However, the skill returned by the skills future was Learning Environment Design. The description of that skill does not applicable to this course, so I am not gonna edit the extracted_skill for this row.

I will however do a check for all courses that has this extracted skill

### Check courses with extracted skill Learning Environment Design

In [29]:
course_codes = df[
    df["extracted_skills"].fillna("").str.contains("Learning Environment Design", case=False, na=False)
]["moduleCode"].tolist()

print(course_codes)

['AN3206', 'BN4102', 'BN4108', 'BN4701', 'CDE2000', 'CDE2001', 'CDE2506', 'CDE2701A', 'CDE2701B', 'CDE4501', 'CDE4501A', 'CDE4501B', 'CE3102', 'CFG1004', 'CFG1500', 'CFG1600', 'CFG1600S', 'CFG3001', 'CH2276', 'CH4660', 'CH4660HM', 'CLC2101', 'CLC3308', 'CM3288NR', 'CM3289NR', 'CS3219', 'CS3242', 'DTK1234A', 'EC4354', 'EC4354HM', 'EC4660', 'EC4660HM', 'EE1001X', 'EE3030B', 'EE4217', 'EG2701A', 'EG2701B', 'EL3259', 'EL4660', 'EL4660HM', 'EN4660', 'EN4660HM', 'ENV2288R', 'ENV2289R', 'ENV3102', 'ENV3288R', 'ENV3289R', 'ES1000', 'ES1103', 'ES2007D', 'EU4660', 'EU4660HM', 'FAS2882D', 'FAS2882F', 'GE2231', 'GE2234', 'GE3252', 'HY4660', 'HY4660HM', 'ID2100', 'ID2106', 'ID2107', 'ID2108', 'ID2109', 'ID2110', 'ID2113', 'ID2114', 'ID2115', 'ID2116', 'ID2118', 'ID3102', 'ID3106', 'ID3107', 'ID3108', 'ID3109', 'ID3110', 'ID3124', 'ID3129', 'ID4105', 'ID4107', 'ID4108', 'ID4109', 'ID4110', 'ID4122', 'IE3250', 'INT3204', 'IPM1104', 'IPM2101', 'IPM3102', 'JS1101E', 'JS3229', 'JS4207', 'JS4207HM', 'JS4

In [30]:
target_skill = "Learning Environment Design"

mask = df["moduleCode"].fillna("").str.startswith("LA")

def remove_skill(skills_text):
    skills = [s.strip() for s in str(skills_text).split("|") if s.strip()]
    skills = [s for s in skills if s.casefold() != target_skill.casefold()]
    return " | ".join(skills)

df.loc[mask, "extracted_skills"] = (
    df.loc[mask, "extracted_skills"]
    .fillna("")
    .apply(remove_skill)
)

In [31]:
print(
    df.loc[
        df["moduleCode"].fillna("").str.startswith("LA")
        & df["extracted_skills"].fillna("").str.contains(target_skill, case=False, na=False),
        ["moduleCode", "extracted_skills"]
    ]
)

Empty DataFrame
Columns: [moduleCode, extracted_skills]
Index: []


In [32]:

df.to_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv", index=False)

# Check CFG courses

In [ ]:
df = pd.read_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv").head()

,moduleCode,title,description,prereqCourseCodes,extracted_skills,extracted_apps_tools,done
0,ACC1701,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
1,ACC1701A,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
2,ACC1701B,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
3,ACC1701C,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...
4,ACC1701D,Accounting for Decision Makers,The course provides an introduction to account...,NaN,Financial Statements Review | Financial Report...,NaN,error: Section 'extracted_apps_and_tools' not ...


In [36]:
cfg_rows = df[df["moduleCode"].astype(str).str.startswith("CFG", na=False)]

In [37]:
print(cfg_rows[["moduleCode", "extracted_skills"]].to_string(index=False))

moduleCode                                                                                                                                                                                             extracted_skills
   CFG1002                                                                                                     Programme Delivery | Business Needs Analysis | Market Specialisation | Project Plan | Strategy Execution
   CFG1003                                                                                                                    Behavioural Finance | Automation Design | Management Decision Making | Debt Restructuring
   CFG1004                                                                                                                                                             Financial Planning | Learning Environment Design
   CFG1500                                                                                                             Learning Environm

I did some manual review and edited the csv manually so there is no code for this edit

# Remove course with description < 20 words
I realised some course description are very short so i decided to remove them

This also removed course with description = Not Applicable

Reason for doing this is because that is the minimum word count recommended by the skills future website. Source: https://jobsandskills.skillsfuture.gov.sg/data-and-tools/tools/skills-extraction-and-comparison-tool

I have removed the courses already, the code is not in this notebook. The code is in `filter_and_extract_module.py`